In [14]:
import pmagpy.pmag as pmag
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import pmagpy.pmag as pmag 
import pmagpy.pmagplotlib as pmagplotlib
import pmagpy.ipmag as ipmag
from pmagpy import pmag, ipmag, pmagplotlib
from pathlib import Path                                                                                                                                                         
import math
import time as time
from statistics import mean,median
import seaborn as sns
import importlib
%matplotlib inline
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from _aux import di2vgp, get_pseudopoles, R_pole_space, P_pole_space

import _aux

# CHECK CARTOPY
has_cartopy, cartopy = pmag.import_cartopy()
if has_cartopy:
    print('Cartopy installed')
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

print("Import Complete")

Cartopy installed
Import Complete


In [15]:
input_filename_path = r"C:\Users\Usuario\Documents\PMAGpythonRotation\data\tec_c.csv"
df = pd.read_csv(input_filename_path,skiprows=0,header=0,skip_blank_lines=True) # build dataframe from Excel input file
df = df.reset_index(drop=True,inplace=False) # reset index of dataframe
N_poles = len(df)

df = df.rename(columns={
            "AgeUpBound": "max_age",
            "AgeLowBound":"min_age"
            })

# if age uncertainties are not given, use ±1 Ma 
df.min_age.fillna((df.age-1.), inplace=True)
df.max_age.fillna((df.age+1.), inplace=True)

# get pole latitudes and longitude
if 'mdec' in df:
    if len(df['mdec']) > 0: 
        mean_DIs = [[df['mdec'][i],df['minc'][i]] for i in range(N_poles)]
        samp_locs = [[df['slat'][i],df['slon'][i]] for i in range(N_poles)]
        paleopoles = di2vgp(di_block=mean_DIs,samp_loc=samp_locs)

    # add to dataframe
    poles_df = pd.DataFrame(paleopoles,columns=['plon','plat'])
    df['plat'] = poles_df['plat']
    df['plon'] = poles_df['plon']
    
    # get reference location from sampling locations
    locations = ipmag.make_di_block(df['slon'].tolist(),df['slat'].to_list())
    loc_princ = pmag.doprinc(locations)
    mean_loc = [loc_princ['dec'],loc_princ['inc']]
    print('Mean sampling location = ', mean_loc)

if 'P95' in df.columns:
    df['A95'] = df['P95']
    df['name'] = df.apply(lambda x: 'pole_at_%1d_Ma' % x['age'], axis=1)
    df['K'] = df['mean_K']
    # EXPORT TO FILE

df = df.dropna(subset=['N'])

df = df.astype({'N':'int'})

df.to_csv('temp_tec.csv', index=None)

print('Number of paleopoles =',N_poles)

Mean sampling location =  [np.float64(356.4752109130354), np.float64(53.84661369253032)]
Number of paleopoles = 7


In [16]:
# Choose reference for computing displacements: 'gapwap', 'custom', 'geopole', 'refpole'
ref_type = 'gapwap'

# Get reference data
if ref_type =='gapwap':
    ref_filename = r'C:\Users\Usuario\Documents\PMAGpythonRotation\data\Reference_database_Vaes_et_al_2023.xlsx'
    ref_df = pd.read_excel(ref_filename,skiprows=2,header=0,usecols='A:AB') # build dataframe from Excel input file
    # if K and A95 are unavailable, replace with estimated value from Cox (1970) formula
    ref_df.K.fillna(ref_df.K_est, inplace=True)
    ref_df.A95.fillna(ref_df.A95_est, inplace=True)
    # convert to northern hemisphere
    ref_df['plat'] = ref_df['plat'].apply(lambda x: x*-1)
    ref_df['plon'] = ref_df['plon'].apply(lambda x: x-180. % 360)
    
    ref_df = ref_df.astype({'N':'int'}) # ensure values for N are integers

    print('Reference datasets = ',len(ref_df))
    print(ref_df.head())

elif ref_type == 'custom':
    ref_filename = 'NEJ.csv'   # read database file
    ref_df = pd.read_csv(ref_filename,skiprows=0,header=0) # build dataframe from Excel input file
    # if age uncertainties are not given, use ±2 Ma 
    ref_df.min_age.fillna((ref_df.age-2.), inplace=True)
    ref_df.max_age.fillna((ref_df.age+2.), inplace=True)
    
    ref_df = ref_df.astype({'N':'int'}) # ensure values for N are integers

    print('Reference datasets = ',len(ref_df))
    print(ref_df.head())
elif ref_type == 'geopole':
    ref_pole = [0,90]
elif ref_type == 'refpole':
    ref_pole = [ref_plon,ref_plat]
    B95 = ref_A95 # is stored as B95 but is simply 95% confidence region of reference pole

Reference datasets =  350
         name  min_age  max_age     age   slat    slon   N  mDec  mInc   k  \
0   Roperch15    0.000    0.005  0.0025 -38.91  288.26  18   NaN   NaN NaN   
1  Dichiara14    0.002    0.008  0.0050  38.59  331.22  12   NaN   NaN NaN   
2    Kissel15    0.000    0.015  0.0075  28.18  343.95  39   NaN   NaN NaN   
3     Salis89    0.008    0.012  0.0100  45.52    2.81   6   NaN   NaN NaN   
4    Tanaka09    0.000    0.021  0.0105 -38.16  176.50  13   NaN   NaN NaN   

   ...       Rlat        Rlon     EP_lat      EP_lon    EP_ang  lithology   f  \
0  ... -85.569348  162.837805  60.350000  -38.740000  0.000715    igneous NaN   
1  ... -88.980260  146.929895  21.037123  -20.423398  0.000626    igneous NaN   
2  ... -86.411049  329.646721   0.000000    0.000000  0.000000    igneous NaN   
3  ... -87.192212   73.014943  21.037037  -20.422853  0.001252    igneous NaN   
4  ... -88.852634   35.682249 -12.764688 -126.480273  0.006459    igneous NaN   

   p_std  Deenen  

In [17]:
### SETTINGS
Nb = 50
ref_window = 10
ref_plate = 301
#ref_type = 'gapwap'

# CHOOSE REFERENCE LOCATION ('one_ref_loc', 'mean_loc', 'samp_loc')
ref_loc_type = 'samp_loc'
one_ref_loc = [140,39] # [lon,lat]

# GET EULER ROTATION POLES
#    if ref_type == 'gapwap':
#        EP_data = np.genfromtxt('Euler_poles_%3d.csv' % ref_plate,skip_header=1,delimiter=',') # load csv file
#    else:
#        EP_data = []

euler_poles_path = r"C:\Users\Usuario\Documents\PMAGpythonRotation\Euler_poles.csv"

ep_all = pd.read_csv(euler_poles_path)

EP_data = ep_all[ep_all['Moving plate ID']==ref_plate][['Moving plate ID','Age (GTS2020)','Latitude','Longitude','Angle','Fixed Plate ID']].dropna().to_numpy()


print(f"Loaded {len(EP_data)} Euler pole entries for plate {ref_plate}")
unique_ages = np.unique(EP_data[:,1])

print(f"Available ages for plate {ref_plate}: {unique_ages}")
print("Settings loaded successfuly.")
print(EP_data.shape)

Loaded 36 Euler pole entries for plate 301
Available ages for plate 301: [  0.      0.773   1.775   2.595   3.596   4.187   5.235   6.023   6.727
   7.537   8.125   9.105   9.786  11.056  12.474  13.739  14.613  15.974
  17.235  18.007  18.636  19.535  33.47   39.24   47.     53.58   57.38
  67.28   79.9    83.65  121.4   126.51  153.44  190.    220.    330.   ]
Settings loaded successfuly.
(36, 6)


In [19]:
# APPLY ALGORITHM TO EACH DATASET

# ---------
importlib.reload(_aux)

def calcRPD(df):
    
    df = df.apply(lambda x: pd.to_numeric(x, errors="coerce")).fillna(0)
    N_poles=int(len(df))
    results=[]
    for i in range(N_poles):
        
        # GET KEY PARAMETERS
        #name, age, min_age, max_age =df["name"][i], df["age"][i], df['min_age'][i],df['max_age'][i]
        age, min_age, max_age, mdec =pd.to_numeric(df["age"][i]), pd.to_numeric(df['min_age'][i]),pd.to_numeric(df['max_age'][i]), pd.to_numeric(df['mdec'][i])
        N_s, min_age_test, max_age_test = pd.to_numeric(df['N'][i]),pd.to_numeric(df['min_age'][i]),pd.to_numeric(df['max_age'][i])
        PP_lon, PP_lat, PP_A95 = df['plon'][i],df['plat'][i],df['A95'][i]
        
        # MODIFY REFERENCE DATA WINDOW
        if (max_age_test-min_age_test) < ref_window:
            max_age_test = df['age'][i]+(ref_window/2.)
            min_age_test = df['age'][i]-(ref_window/2.)
        
        # GET REFERENCE LOCATION
        if ref_loc_type == 'one_ref_loc':
            ref_loc = one_ref_loc
        elif ref_loc_type == 'mean_loc':
            ref_loc = mean_loc
        else:
            ref_loc = [df['slon'][i],df['slat'][i]]
            
        #print('')
        print('Entry number = %2d ; N = %1d, age range = %1.1f - %1.1f' % (i,N_s,min_age_test,max_age_test))
        
        # ------
        if ref_type == 'gapwap' or ref_type == 'custom':
            # GET PSEUDOPOLES    
            ppoles = []

            # SELECT PALEOPOLES FROM REFERENCE DATABASE
            df_ref_poles = ref_df[(ref_df.min_age <= max_age_test) & (ref_df.max_age >= min_age_test)].copy()
            df_ref_poles.reset_index()

            N_s = int(N_s)

            # GET PSEUDOPOLES
            ppoles = [get_pseudopoles(df=df_ref_poles,age_min=min_age_test,age_max=max_age_test,N_s=N_s,EP_data=EP_data) for k in range(Nb)]

            # COMPUTE REFERENCE POLE AND DIRECTION
            ppoles_mean = pmag.fisher_mean(ppoles) # compute mean of pseudopoles
            ref_pole = [ppoles_mean['dec'],ppoles_mean['inc']] # store reference mean

            # COMPUTE ANGULAR DISTANCE OF PSEUDOPOLES TO REFERENCE MEAN
            D_ppoles=[]
            for j in range(Nb):
                ang_distance=pmag.angle(ppoles[j],ref_pole)
                D_ppoles.append(ang_distance[0])

            # COMPUTE STATISTICS
            D_ppoles.sort() # sort angular distances to mean pole
            ind_95perc=int(0.95*Nb)-1 # find index of 95% percentile
            B95 = D_ppoles[ind_95perc] # get B95
            
            ref_N_values = [pseudopole[2] for pseudopole in ppoles]
            ref_N_mean = np.mean(ref_N_values) # compute mean N of simulated reference poles
            ref_K_values = [pseudopole[3] for pseudopole in ppoles]
            ref_K_mean = np.mean(ref_K_values) # compute mean K of simulated reference poles
            
        elif ref_type == 'geopole' or ref_type == 'refpole':
            ppoles_mean = {'dec': ref_pole[0], 'inc': ref_pole[1]}
            B95 = 0
            ref_N_mean = 0
            ref_K_mean = 0
            
        # ---------------
        # compute rotation in pole-space
        R,dr = R_pole_space(ref_pole,B95,[PP_lon,PP_lat],PP_A95,ref_loc)

        if R-dr>0 or R+dr<0: # check if lower confidence limit of rotation is larger than zero
            RS = True
        else:
            RS = False

        if 'EI_unc_plat' in df:
            if df['EI_unc_plat'][i]>0:
                PP_A95 = df['EI_unc_plat'][i]

        P,dp = P_pole_space(ref_pole,B95,[PP_lon,PP_lat],PP_A95,ref_loc)
        L = P*-1
        dl = dp

        if P-dp>0 or P+dp<0: # check if lower confidence limit of rotation is larger than zero
            LS = True
        else:
            LS = False

        ref_DI = pmag.vgp_di(ref_pole[1],ref_pole[0],ref_loc[1],ref_loc[0]) # compute reference direction

        # Append the result
        results.append({
            #"name": name,
            "age": age, "max_age":max_age, "min_age": min_age, "N":N_poles,
            "R": R, "delta_R": dr, "R_sig": RS, 
            "L": L, "delta_L": dl, "L_sig": LS,
            "B95": B95, "A95": PP_A95,
            "plat": ppoles_mean['inc'], "plon": ppoles_mean['dec'], 
            "slat": ref_loc[1], "slon": ref_loc[0],
            "ref_dec": ref_DI[0], "dec": mdec, "ref_inc": ref_DI[1],
            "ref_mean_N": ref_N_mean, "ref_mean_K": ref_K_mean})

    print ("Done!")
    return results
df = pd.read_csv(r"C:\Users\Usuario\Documents\PMAGpythonRotation\cur_calc_plot\temp_tec.csv")
results = pd.DataFrame(calcRPD(df))

results_folder = Path(r"C:\Users\Usuario\Documents\PMAGpythonRotation\results")

results_folder.mkdir(parents=True, exist_ok=True)

# Define the CSV filename
results_file = results_folder / "results.csv"

# Save the DataFrame
results.to_csv(results_file, index=False)

print(f"Results saved to: {results_file}")



Entry number =  0 ; N = 75, age range = 279.0 - 293.0
Entry number =  1 ; N = 153, age range = 254.2 - 264.3
Entry number =  2 ; N = 64, age range = 285.0 - 309.0
Entry number =  3 ; N = 13, age range = 318.0 - 358.0
Entry number =  4 ; N = 5, age range = 295.0 - 305.0
Entry number =  5 ; N = 27, age range = 208.0 - 227.0
Entry number =  6 ; N = 138, age range = 241.5 - 251.5
Done!
Results saved to: C:\Users\Usuario\Documents\PMAGpythonRotation\results\results.csv
